# 🧠 Redes Neuronales - Implementación desde CERO

## Objetivos
- Entender neuronas artificiales
- Implementar una red neuronal desde cero
- Backpropagation
- Entrenar en clasificación binaria

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Agregar el directorio raíz al path de manera robusta
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar que las utilidades se pueden importar
try:
    from utils.plot_utils import plot_decision_boundary, plot_learning_curve
    print("✅ Entorno configurado correctamente")
    print(f"📁 Raíz del proyecto: {project_root}")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("\n💡 Soluciones:")
    print("   1. Ejecuta 'pip install -e .' desde la raíz del proyecto")
    print("   2. O inicia Jupyter desde la raíz: cd ML-FROM-ZERO-PYTHON && jupyter notebook")
    print(f"\n🔍 Error detallado: {e}")
    raise

## 1. Teoría de Redes Neuronales

### Neurona Artificial
$$z = w^T x + b$$
$$a = \sigma(z)$$

### Red Neuronal (arquitectura simple):
- **Capa de entrada**: Features (X)
- **Capa oculta**: Neuronas que aprenden representaciones
- **Capa de salida**: Predicción

### Forward Propagation:
$$Z^{[1]} = W^{[1]} X + b^{[1]}$$
$$A^{[1]} = \sigma(Z^{[1]})$$
$$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$$
$$A^{[2]} = \sigma(Z^{[2]})$$

### Backpropagation:
Calcular gradientes usando la regla de la cadena para actualizar pesos.

## 2. Implementación desde CERO

In [ ]:
class RedNeuronal:
    """
    Red Neuronal simple (1 capa oculta) desde cero.
    """
    
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.learning_rate = learning_rate
        
        # Inicializar pesos aleatoriamente
        np.random.seed(42)
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))
        
        self.loss_history = []
    
    def _sigmoid(self, z):
        """Función de activación sigmoide"""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def _sigmoid_derivative(self, a):
        """Derivada de sigmoide"""
        return a * (1 - a)
    
    def forward(self, X):
        """
        Forward propagation.
        
        Returns:
            Z1, A1, Z2, A2
        """
        # Capa oculta
        Z1 = np.dot(X, self.W1) + self.b1
        A1 = self._sigmoid(Z1)
        
        # Capa de salida
        Z2 = np.dot(A1, self.W2) + self.b2
        A2 = self._sigmoid(Z2)
        
        return Z1, A1, Z2, A2
    
    def backward(self, X, y, Z1, A1, Z2, A2):
        """
        Backpropagation - calcula gradientes.
        """
        m = X.shape[0]
        
        # Gradientes de la capa de salida
        dZ2 = A2 - y.reshape(-1, 1)
        dW2 = (1/m) * np.dot(A1.T, dZ2)
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
        
        # Gradientes de la capa oculta
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self._sigmoid_derivative(A1)
        dW1 = (1/m) * np.dot(X.T, dZ1)
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)
        
        return dW1, db1, dW2, db2
    
    def update_parameters(self, dW1, db1, dW2, db2):
        """Actualiza pesos usando gradient descent"""
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
    
    def _binary_cross_entropy(self, y_true, y_pred):
        """Binary Cross-Entropy Loss"""
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    def fit(self, X, y, epochs=1000, verbose=True):
        """Entrena la red neuronal"""
        for epoch in range(epochs):
            # Forward propagation
            Z1, A1, Z2, A2 = self.forward(X)
            
            # Calcular loss
            loss = self._binary_cross_entropy(y, A2)
            self.loss_history.append(loss)
            
            # Backpropagation
            dW1, db1, dW2, db2 = self.backward(X, y, Z1, A1, Z2, A2)
            
            # Actualizar parámetros
            self.update_parameters(dW1, db1, dW2, db2)
            
            # Imprimir progreso
            if verbose and (epoch + 1) % 100 == 0:
                print(f"Época {epoch+1}/{epochs}, Loss: {loss:.4f}")
        
        return self
    
    def predict_proba(self, X):
        """Predice probabilidades"""
        _, _, _, A2 = self.forward(X)
        return A2.flatten()
    
    def predict(self, X, threshold=0.5):
        """Predice clases"""
        return (self.predict_proba(X) >= threshold).astype(int)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

## 3. Ejemplo: Clasificación No Lineal

In [ ]:
from sklearn.datasets import make_moons

# Generar datos no lineales (forma de luna)
X, y = make_moons(n_samples=300, noise=0.2, random_state=42)

# Visualizar
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='red', label='Clase 0', edgecolors='k')
plt.scatter(X[y==1, 0], X[y==1, 1], c='blue', label='Clase 1', edgecolors='k')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos No Lineales (Moons)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Crear y entrenar red neuronal
# input_size=2 (2 features), hidden_size=4 (4 neuronas ocultas), output_size=1
nn = RedNeuronal(input_size=2, hidden_size=4, output_size=1, learning_rate=0.5)
nn.fit(X, y, epochs=2000, verbose=True)

# Evaluar
accuracy = nn.score(X, y)
print(f"\nAccuracy: {accuracy:.4f}")

In [ ]:
# Visualizar frontera de decisión
plot_decision_boundary(X, y, nn, title="Red Neuronal - Frontera de Decisión No Lineal")
plt.show()

In [ ]:
# Curva de aprendizaje
plt.figure(figsize=(10, 6))
plt.plot(nn.loss_history, linewidth=2)
plt.xlabel('Época')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Curva de Aprendizaje - Red Neuronal')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Loss inicial: {nn.loss_history[0]:.4f}")
print(f"Loss final: {nn.loss_history[-1]:.4f}")

## 4. Comparación con Regresión Logística

La regresión logística solo puede aprender fronteras lineales.

In [ ]:
# Importar nuestra Regresión Logística
sys.path.append('../../notebooks/algoritmos-supervisados')

from sklearn.linear_model import LogisticRegression

# Entrenar regresión logística
lr = LogisticRegression()
lr.fit(X, y)

# Comparar fronteras de decisión
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Regresión Logística
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()])
Z_lr = Z_lr.reshape(xx.shape)

axes[0].contourf(xx, yy, Z_lr, alpha=0.3, cmap='RdBu')
axes[0].scatter(X[y==0, 0], X[y==0, 1], c='red', edgecolors='k')
axes[0].scatter(X[y==1, 0], X[y==1, 1], c='blue', edgecolors='k')
axes[0].set_title(f'Regresión Logística\nAccuracy: {lr.score(X, y):.4f}')

# Red Neuronal
Z_nn = nn.predict(np.c_[xx.ravel(), yy.ravel()])
Z_nn = Z_nn.reshape(xx.shape)

axes[1].contourf(xx, yy, Z_nn, alpha=0.3, cmap='RdBu')
axes[1].scatter(X[y==0, 0], X[y==0, 1], c='red', edgecolors='k')
axes[1].scatter(X[y==1, 0], X[y==1, 1], c='blue', edgecolors='k')
axes[1].set_title(f'Red Neuronal\nAccuracy: {nn.score(X, y):.4f}')

plt.tight_layout()
plt.show()

print("La Red Neuronal puede aprender patrones no lineales!")

## 🎯 Ejercicio: Experimentar con Arquitectura

Prueba diferentes configuraciones.

In [ ]:
# TU CÓDIGO AQUÍ
# Prueba:
# 1. Diferente número de neuronas ocultas (2, 8, 16)
# 2. Diferentes learning rates (0.1, 1.0, 2.0)
# 3. Diferentes épocas
# 4. Visualiza cómo cambia la frontera de decisión

print("Experimenta con la arquitectura aquí")

## 🎓 Resumen

- ✅ Redes neuronales aprenden representaciones no lineales
- ✅ Forward propagation: Calcular predicciones
- ✅ Backpropagation: Calcular gradientes
- ✅ Gradient descent: Actualizar pesos
- ✅ Capas ocultas permiten aprender patrones complejos

### Conceptos Avanzados:
- Más capas = Deep Learning
- Funciones de activación: ReLU, tanh, etc.
- Regularización: Dropout, L2
- Optimizadores: Adam, RMSprop

### ¡Felicidades! Has implementado una Red Neuronal desde CERO